# 🔍 Fairness Analysis Notebook (with Fairlearn & AIF360)

This notebook demonstrates fairness analysis using two popular fairness toolkits:
- **Fairlearn**: For computing group fairness metrics via `MetricFrame`
- **AIF360**: For computing disparate impact on structured datasets

## Setup

Install the required fairness libraries if not already available.

## 1. Data Preparation

Create a synthetic dataset with a known gender-based bias in the outcome.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'income': np.random.normal(50000, 15000, n),
})

df['outcome'] = (df['income'] > 50000).astype(int)
df.loc[df['gender'] == 'Female', 'outcome'] -= 0.1
df['outcome'] = df['outcome'].clip(0, 1).astype(int)

df.head()

## 2. Fairlearn Analysis

Train a `RandomForestClassifier` and evaluate selection rates across gender groups
using Fairlearn's `MetricFrame`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from fairlearn.metrics import MetricFrame, selection_rate

X = df[['age', 'income']]
y = df['outcome']
A = df['gender']

X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, A, test_size=0.3, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mf = MetricFrame(
    metrics={'selection_rate': selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test
)

print("📊 Selection rate by group:")
print(mf.by_group)

## 3. AIF360 Analysis

Compute **Disparate Impact** using AIF360's `BinaryLabelDatasetMetric`.

In [ ]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

dataset = BinaryLabelDataset(
    df=df,
    label_names=['outcome'],
    protected_attribute_names=['gender']
)

metric = BinaryLabelDatasetMetric(
    dataset,
    privileged_groups=[{'gender': 1}],
    unprivileged_groups=[{'gender': 0}]
)

print("⚖️ Disparate Impact:", metric.disparate_impact())